# Churn prediction — clean build

Structured on the eight steps in *Hands-On Machine Learning* (Géron), Chapter 2.

1. Look at the big picture
2. Get the data
3. Discover and visualise
4. Prepare the data
5. Shortlist promising models
6. Fine-tune
7. Evaluate on the test set — once
8. Save

## 1. Look at the big picture

**Task.** Predict which credit card customers will close their card, so the marketing team can make a retention offer.

**Type.** Supervised, binary classification, batch learning.

**Measure.** F1 on the churn class. Accuracy is unusable — 80% of customers stay, so predicting "nobody closes" scores 80% and catches nobody.

**Assumption.** The prediction is used to choose who gets an offer, so both missing a churner and wasting an offer carry cost. F1 balances the two.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_predict, RandomizedSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

RANDOM_STATE = 42

## 2. Get the data

The test set is created **now**, before anything is looked at. Géron: *"your brain is an amazing pattern detection system... if you look at the test set, you may stumble upon some seemingly interesting pattern that leads you to select a particular kind of model."*

Stratified so both halves keep the same churn rate.

In [ ]:
try:
    from google.colab import files
    DATA = list(files.upload().keys())[0]
except ImportError:
    DATA = "../data/raw/bank_churn_dataset.csv"

churn = pd.read_csv(DATA)

train_set, test_set = train_test_split(
    churn, test_size=0.2, stratify=churn["churned"], random_state=RANDOM_STATE)

print(len(train_set), "train /", len(test_set), "test")
print("churn rate:", round(train_set["churned"].mean(), 4),
      "/", round(test_set["churned"].mean(), 4))

## 3. Discover and visualise

Training set only. The test set is not touched again until step 7.

In [ ]:
explore = train_set.copy()

print(explore.shape)
print("missing:", explore.isnull().sum().sum(), " duplicates:", explore.duplicated().sum())
explore.head()

In [ ]:
purchase_cols = [f"purchase_month_{i}" for i in range(1, 7)]

monthly = explore.groupby("churned")[purchase_cols].mean().T
monthly.columns = ["stayed", "closed"]
monthly.round(0)

In [ ]:
explore.corr(numeric_only=True)["churned"].abs().sort_values(ascending=False).head(12).round(3)

## 4. Prepare the data

Written as transformers so the same steps apply to the training set, the test set and any future customer.

Two things a linear model cannot derive from the raw columns, so they are built explicitly:

- **ratios and squares** — `purchase_volatility`, `payment_ratio`, `spend_to_salary`
- **products** — the spending trend multiplied by each other feature

A slope or an average is a weighted sum of columns already present, so a linear model can construct those itself. They are not built here.

Clipping the extreme 1% and scaling are both fitted inside the pipeline, on training folds only.

In [ ]:
PURCHASE = [f"purchase_month_{i}" for i in range(1, 7)]
PAYMENT   = [f"payment_month_{i}" for i in range(1, 7)]
MONTHS    = np.array([1, 2, 3, 4, 5, 6])

PARTNERS = ["relationship_depth", "payment_ratio", "purchase_volatility",
            "missed_loan_payment_ever", "recent_purchases", "responsibility",
            "spend_to_salary", "has_other_credit_cards",
            "salary_lands_in_bank", "iscore"]


class AddFeatures(BaseEstimator, TransformerMixin):
    """Build the columns a linear model cannot derive on its own."""

    def fit(self, X, y=None):
        d = self._build(X)
        self.rank_ref_ = {c: np.sort(d[c].values.astype(float))
                          for c in ["purchase_slope"] + PARTNERS}
        return self

    def transform(self, X):
        d = self._build(X)
        s = self._pct(d["purchase_slope"], "purchase_slope")
        for p in PARTNERS:
            d["slope_x_" + p] = s * self._pct(d[p], p)
        return d.drop(columns=PAYMENT + ["is_paying_old_loan"])

    def _build(self, X):
        d = X.copy()
        d["purchase_slope"]      = d[PURCHASE].apply(
            lambda r: np.polyfit(MONTHS, r.values, 1)[0], axis=1)
        d["recent_purchases"]    = d[PURCHASE[3:]].mean(axis=1)
        d["purchase_volatility"] = d[PURCHASE].std(axis=1) / d[PURCHASE].mean(axis=1)
        d["payment_ratio"]       = d[PAYMENT].sum(axis=1) / d[PURCHASE].sum(axis=1)
        d["spend_to_salary"]     = d[PURCHASE].mean(axis=1) / d["salary"]
        d["responsibility"]      = d["married"] + d["has_dependents"]
        d["relationship_depth"]  = (d["salary_lands_in_bank"]
                                    + (1 - d["has_other_credit_cards"])
                                    + (1 - d["missed_loan_payment_ever"]))
        return d

    def _pct(self, values, column):
        ref = self.rank_ref_[column]
        return np.searchsorted(ref, np.asarray(values, dtype=float), side="right") / len(ref)


class ClipExtremes(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower, self.upper = lower, upper

    def fit(self, X, y=None):
        Xv = np.asarray(X, dtype=float)
        self.low_, self.high_ = (np.quantile(Xv, self.lower, axis=0),
                                 np.quantile(Xv, self.upper, axis=0))
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.low_, self.high_)

In [ ]:
X_train = train_set.drop(columns=["churned", "customer_id", "gender"])
y_train = train_set["churned"].copy()

X_test = test_set.drop(columns=["churned", "customer_id", "gender"])
y_test = test_set["churned"].copy()

added = AddFeatures().fit(X_train)
sample = added.transform(X_train.head())

num_cols = [c for c in sample.columns if c != "employment_sector"]
cat_cols = ["employment_sector"]

prep = Pipeline([
    ("features", AddFeatures()),
    ("columns", ColumnTransformer([
        ("num", Pipeline([("clip", ClipExtremes()), ("scale", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])),
])

print(prep.fit_transform(X_train, y_train).shape)

## 5. Shortlist promising models

Géron: *"Train many quick-and-dirty models from different categories using standard parameters... for each model, use N-fold cross-validation."*

Four families, default settings, five folds. The threshold is chosen on the cross-validated predictions, so no model is judged on a single split.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def best_f1(y_true, prob):
    best, cut = 0, 0
    for c in np.arange(0.05, 0.85, 0.005):
        f = f1_score(y_true, (prob >= c).astype(int), zero_division=0)
        if f > best:
            best, cut = f, c
    return best, cut


candidates = {
    "logistic":      LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "random forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "naive bayes":   GaussianNB(),
}

rows = []
for name, clf in candidates.items():
    model = Pipeline([("prep", prep), ("clf", clf)])
    p = cross_val_predict(model, X_train, y_train, cv=folds, method="predict_proba")[:, 1]
    f, cut = best_f1(y_train, p)
    rows.append({"model": name, "AUC": round(roc_auc_score(y_train, p), 4),
                 "F1": round(f, 4), "cut": round(cut, 3)})

pd.DataFrame(rows).set_index("model")

## 6. Fine-tune

Géron: *"Unless there are very few hyperparameter values to explore, prefer random search over grid search"*, and *"treat your data transformation choices as hyperparameters."*

So the amount of clipping is searched alongside the model's own settings.

In [ ]:
model = Pipeline([("prep", prep),
                  ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])

search = RandomizedSearchCV(
    model,
    {"clf__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3],
     "prep__columns__num__clip__lower": [0.0, 0.005, 0.01, 0.02]},
    n_iter=20, scoring="roc_auc", cv=folds,
    random_state=RANDOM_STATE, n_jobs=-1)

search.fit(X_train, y_train)

print("best AUC:", round(search.best_score_, 4))
print(search.best_params_)

In [ ]:
final_model = search.best_estimator_

p_cv = cross_val_predict(final_model, X_train, y_train, cv=folds, method="predict_proba")[:, 1]
CV_F1, CUT = best_f1(y_train, p_cv)

print("cross-validated AUC:", round(roc_auc_score(y_train, p_cv), 4))
print("cross-validated F1: ", round(CV_F1, 4), "at cut", round(CUT, 3))

### Error analysis

Géron: *"You should also look at the specific errors that your system makes."*

In [ ]:
missed = (y_train == 1) & (p_cv < CUT)
stayed = y_train == 0

built = AddFeatures().fit(X_train).transform(X_train)
numeric = built.select_dtypes("number")

pd.DataFrame({
    "missed churners": numeric[missed.values].mean(),
    "customers who stayed": numeric[stayed.values].mean(),
}).assign(ratio=lambda t: (t.iloc[:, 0] / t.iloc[:, 1]).round(2)).sort_values("ratio").round(2)

## 7. Evaluate on the test set — once

Géron: *"Don't tweak your model after measuring the generalization error: you would just start overfitting the test set."*

Everything above used training data only. This cell runs once.

In [ ]:
final_model.fit(X_train, y_train)

p_test = final_model.predict_proba(X_test)[:, 1]
pred = (p_test >= CUT).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

print("AUC      :", round(roc_auc_score(y_test, p_test), 4))
print("F1       :", round(f1_score(y_test, pred), 4))
print("precision:", round(tp / (tp + fp), 3))
print("recall   :", round(tp / (tp + fn), 3))
print()
print("caught", tp, "| missed", fn, "| false alarms", fp)

## 8. Save

The column order and the threshold are saved with the model, so anything scoring a customer later cannot get them wrong.

In [ ]:
import joblib

joblib.dump({"model": final_model, "threshold": CUT,
             "cv_f1": CV_F1, "cv_auc": round(roc_auc_score(y_train, p_cv), 4)},
            "churn_model.pkl")

print("saved")